# RelBench rel-stack Dataset Preparation for JRN Estimation

This notebook demonstrates the **RelBench rel-stack dataset** (Stack Exchange stats-exchange) prepared for **Join Reproduction Number (JRN)** estimation experiments.

**What this artifact does:**
- Loads pre-computed relational dataset with 7 tables (users, posts, comments, votes, badges, postLinks, postHistory)
- Explores the relational schema, foreign key relationships, and table statistics
- Analyzes per-FK-join statistics: fan-out distributions, coverage, cardinality ratios
- Examines multi-hop join chains with cumulative cardinality for JRN multiplicative compounding analysis
- Reviews task definitions (user-engagement, user-badge, post-votes, user-post-comment, post-post-related)
- Visualizes fan-out distributions and cardinality across the relational graph

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# All packages used are pre-installed on Colab; install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0', 'tabulate==0.9.0')

In [ ]:
import json
import math
import os
from collections import Counter
from functools import reduce

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tabulate import tabulate

## Data Loading

Load the pre-computed RelBench rel-stack dataset from GitHub (with local fallback).

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-bc07ab-join-reproduction-number-epidemiology-in/main/dataset_iter1_relbench_rel_st/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded dataset with {len(data['datasets'])} dataset(s)")
print(f"Metadata keys: {list(data['metadata'].keys())}")
print(f"Total examples: {len(data['datasets'][0]['examples'])}")

## Configuration

Tunable parameters for the analysis. Start with minimum values for testing.

In [ ]:
# --- Config ---
# Number of example rows to display per table
MAX_EXAMPLES_DISPLAY = 3
# Number of histogram bins for fan-out visualization
HISTOGRAM_BINS = 10
# Top N joins to highlight in analysis
TOP_N_JOINS = 5
# Temporal split timestamps (from original script)
VAL_TIMESTAMP = "2019-01-01"
TEST_TIMESTAMP = "2021-01-01"

## Relational Schema Exploration

Examine the 7 tables in the rel-stack dataset: their primary keys, foreign keys, columns, and row counts.

In [ ]:
meta = data["metadata"]["rel_stack"]
schema = meta["schema"]

print(f"Total rows in FULL dataset: {meta['total_rows_full']:,}")
print(f"Total tables: {meta['total_tables']}")
print(f"Total columns: {meta['total_columns']}")
print()

# Build schema summary table
schema_rows = []
for tname, tinfo in schema.items():
    fk_list = list(tinfo.get("fkey_col_to_pkey_table", {}).items())
    fk_str = ", ".join(f"{fk}->{pk}" for fk, pk in fk_list) if fk_list else "(none)"
    schema_rows.append([
        tname,
        tinfo["pkey_col"],
        tinfo["time_col"] or "-",
        tinfo["num_rows"],
        tinfo["num_cols"],
        fk_str,
    ])

print(tabulate(
    schema_rows,
    headers=["Table", "PK", "Time Col", "Rows", "Cols", "Foreign Keys"],
    tablefmt="github",
    intfmt=",",
))

## Per-FK Join Statistics

Analyze fan-out distributions, coverage, and cardinality ratios for all 11 foreign key relationships computed on the FULL 4.2M-row dataset.

In [ ]:
join_stats = meta["join_statistics"]
print(f"Total FK relationships: {len(join_stats)}")
print()

# Build join statistics summary table
join_rows = []
for join_key, stats in join_stats.items():
    fo = stats["fan_out_stats"]
    join_rows.append([
        join_key,
        f"{stats['child_rows_total']:,}",
        f"{stats['parent_rows_total']:,}",
        f"{stats['cardinality_ratio']:.2f}",
        f"{fo['mean']:.2f}",
        f"{fo['median']:.1f}",
        f"{fo['max']:,}",
        f"{stats['parent_coverage']:.1%}",
        f"{stats['fk_null_ratio']:.1%}",
    ])

# Sort by cardinality ratio descending
join_rows.sort(key=lambda r: float(r[3]), reverse=True)

print(tabulate(
    join_rows[:TOP_N_JOINS],
    headers=["FK Relationship", "Child Rows", "Parent Rows", "Card. Ratio",
             "Fan-out Mean", "Median", "Max", "Coverage", "Null%"],
    tablefmt="github",
))
print(f"\n(Showing top {TOP_N_JOINS} by cardinality ratio; {len(join_rows)} total)")

## Multi-Hop Join Chains

Examine multi-hop join chains with cumulative cardinality. The JRN framework models how cardinality compounds multiplicatively across chained joins.

In [ ]:
multi_hop = meta["multi_hop_chains"]
print(f"Total multi-hop chains documented: {len(multi_hop)}")
print()

chain_rows = []
for chain in multi_hop:
    hops_str = " -> ".join(chain["hops"])
    fo_str = " x ".join(f"{fo:.2f}" for fo in chain["individual_fan_outs"])
    chain_rows.append([
        chain["depth"],
        hops_str,
        fo_str,
        f"{chain['cumulative_cardinality']:.2f}",
    ])

# Sort by cumulative cardinality descending
chain_rows.sort(key=lambda r: float(r[3]), reverse=True)

print(tabulate(
    chain_rows,
    headers=["Depth", "Join Chain", "Individual Fan-outs", "Cumulative Cardinality"],
    tablefmt="github",
))

## Task Definitions

The rel-stack dataset includes 5 prediction tasks spanning binary classification, regression, and link prediction.

In [ ]:
tasks = meta["tasks"]
print(f"Number of tasks: {len(tasks)}")
print()

task_rows = []
for task_name, info in tasks.items():
    entity = info.get("entity_table", info.get("src_entity_table", "?"))
    target = info.get("target_col", "-")
    train_n = info.get("train_rows", "?")
    val_n = info.get("val_rows", "?")
    test_n = info.get("test_rows", "?")
    task_rows.append([
        task_name,
        info.get("task_type", "?").replace("TaskType.", ""),
        entity,
        target,
        f"{train_n:,}" if isinstance(train_n, int) else str(train_n),
        f"{val_n:,}" if isinstance(val_n, int) else str(val_n),
        f"{test_n:,}" if isinstance(test_n, int) else str(test_n),
    ])

print(tabulate(
    task_rows,
    headers=["Task", "Type", "Entity Table", "Target", "Train", "Val", "Test"],
    tablefmt="github",
))

# Show temporal splits
splits = meta.get("temporal_splits", {})
print(f"\nTemporal splits: val >= {splits.get('val_timestamp', '?')}, test >= {splits.get('test_timestamp', '?')}")

## Example Rows Analysis

Examine the subsampled example rows across all 7 tables, including their temporal fold assignments and foreign key references.

In [ ]:
examples = data["datasets"][0]["examples"]

# Count examples per table and per fold
table_counts = Counter(ex["metadata_table"] for ex in examples)
fold_counts = Counter(ex["metadata_fold"] for ex in examples)
fold_names = {0: "train", 1: "val", 2: "test"}

print("Examples per table:")
for t, c in sorted(table_counts.items()):
    print(f"  {t}: {c}")
print(f"\nExamples per fold:")
for f, c in sorted(fold_counts.items()):
    print(f"  {fold_names.get(f, f'fold_{f}')}: {c}")

# Show a few examples per table
print(f"\n--- Sample rows (up to {MAX_EXAMPLES_DISPLAY} per table) ---")
shown = Counter()
for ex in examples:
    t = ex["metadata_table"]
    if shown[t] >= MAX_EXAMPLES_DISPLAY:
        continue
    shown[t] += 1
    input_data = json.loads(ex["input"])
    fkeys = json.loads(ex["metadata_fkeys"])
    # Truncate long text values for display
    for k, v in input_data.items():
        if isinstance(v, str) and len(v) > 80:
            input_data[k] = v[:77] + "..."
    print(f"\n[{t}] pk={ex['metadata_pk']} fold={fold_names.get(ex['metadata_fold'], '?')}")
    print(f"  FK refs: {fkeys}")
    print(f"  Fields: {list(input_data.keys())}")

## Visualization: Fan-out Distributions and JRN Compounding

Visualize the key relational statistics that drive JRN estimation:
1. **Mean fan-out** per FK relationship (how many child rows per parent)
2. **Cumulative cardinality** for multi-hop chains (multiplicative compounding)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Plot 1: Mean fan-out per FK relationship ---
ax = axes[0, 0]
labels = []
means = []
for join_key, stats in join_stats.items():
    short_label = join_key.split(" -> ")[0]  # e.g. "comments.UserId"
    labels.append(short_label)
    means.append(stats["fan_out_stats"]["mean"])
order = np.argsort(means)[::-1]
ax.barh([labels[i] for i in order], [means[i] for i in order], color="steelblue")
ax.set_xlabel("Mean Fan-out")
ax.set_title("Mean Fan-out per FK Relationship")
ax.grid(axis="x", alpha=0.3)

# --- Plot 2: Cardinality ratio per FK ---
ax = axes[0, 1]
card_ratios = []
card_labels = []
for join_key, stats in join_stats.items():
    short_label = join_key.split(" -> ")[0]
    card_labels.append(short_label)
    card_ratios.append(stats["cardinality_ratio"])
order2 = np.argsort(card_ratios)[::-1]
ax.barh([card_labels[i] for i in order2], [card_ratios[i] for i in order2], color="coral")
ax.set_xlabel("Cardinality Ratio (child_rows / parent_rows)")
ax.set_title("Cardinality Ratio per FK Relationship")
ax.grid(axis="x", alpha=0.3)

# --- Plot 3: Multi-hop cumulative cardinality ---
ax = axes[1, 0]
chain_labels = []
cum_cards = []
depths = []
for chain in multi_hop:
    lbl = " -> ".join(h.split(" -> ")[0].split(".")[0] for h in chain["hops"])
    chain_labels.append(lbl)
    cum_cards.append(chain["cumulative_cardinality"])
    depths.append(chain["depth"])
colors = ["#2196F3" if d == 2 else "#FF5722" for d in depths]
order3 = np.argsort(cum_cards)[::-1]
ax.barh(
    [chain_labels[i] for i in order3],
    [cum_cards[i] for i in order3],
    color=[colors[i] for i in order3],
)
ax.set_xlabel("Cumulative Cardinality")
ax.set_title("Multi-Hop Join Chain Cardinality")
ax.grid(axis="x", alpha=0.3)
# Legend
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color="#2196F3", label="2-hop"), Patch(color="#FF5722", label="3-hop")],
          loc="lower right")

# --- Plot 4: Parent coverage vs fan-out mean (scatter) ---
ax = axes[1, 1]
coverages = []
fan_means = []
scatter_labels = []
for join_key, stats in join_stats.items():
    coverages.append(stats["parent_coverage"])
    fan_means.append(stats["fan_out_stats"]["mean"])
    scatter_labels.append(join_key.split(" -> ")[0])
ax.scatter(coverages, fan_means, s=100, c="green", alpha=0.7, edgecolors="black")
for i, lbl in enumerate(scatter_labels):
    ax.annotate(lbl.split(".")[1], (coverages[i], fan_means[i]),
                fontsize=7, ha="left", va="bottom")
ax.set_xlabel("Parent Coverage")
ax.set_ylabel("Mean Fan-out")
ax.set_title("Coverage vs Fan-out")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("relstack_jrn_analysis.png", dpi=100, bbox_inches="tight")
plt.show()
print("Figure saved to relstack_jrn_analysis.png")

## Summary

Print a concise summary of the dataset's key statistics for JRN estimation.

In [ ]:
print("=" * 60)
print("DATASET SUMMARY: RelBench rel-stack for JRN Estimation")
print("=" * 60)
print(f"  Source: {data['metadata']['source']}")
print(f"  License: {data['metadata']['license']}")
print(f"  Total rows (full): {meta['total_rows_full']:,}")
print(f"  Tables: {meta['total_tables']}")
print(f"  Columns: {meta['total_columns']}")
print(f"  FK relationships: {len(join_stats)}")
print(f"  Multi-hop chains: {len(multi_hop)}")
print(f"  Tasks: {len(tasks)}")
print(f"  Demo examples: {len(examples)}")
print()

# Key JRN insights
max_chain = max(multi_hop, key=lambda c: c["cumulative_cardinality"])
max_fo = max(join_stats.items(), key=lambda x: x[1]["fan_out_stats"]["mean"])
print("Key JRN insights:")
print(f"  Highest single-hop fan-out: {max_fo[0]} (mean={max_fo[1]['fan_out_stats']['mean']:.2f})")
print(f"  Highest cumulative cardinality: {' -> '.join(max_chain['hops'])} ({max_chain['cumulative_cardinality']:.2f})")
print(f"  Max chain depth: {max(c['depth'] for c in multi_hop)}-hop")
print()
print("This dataset enables studying how join cardinality compounds")
print("multiplicatively across relational chains - the core JRN phenomenon.")